In [ ]:
!pip install google-api-python-client

In [2]:
from googleapiclient.discovery import build

In [16]:
import os
from dotenv import load_dotenv

# .env 파일의 환경 변수를 로드합니다.
load_dotenv()

# 환경 변수를 변수에 할당합니다.
api_key = os.getenv("YOUTUBE_API_KEY")

In [17]:
youtube = build("youtube", "v3", developerKey=api_key)
youtube

In [7]:
query = '크보빵'

request = youtube.search().list(
    q=query,
    part="snippet",
    maxResults=10,
    # order="date",
    type="video"
)
response = request.execute()
videos = []
for item in response["items"]:
    video_id = item["id"]["videoId"]
    title = item["snippet"]["title"]
    videos.append((video_id, title))

videos

[('1jIdHHDvxXk', '크보빵 9종 리뷰 #KBO빵'),
 ('sH2hApEEgpo', '빵의도시 &#39;대전&#39;의 자존심을 건 크보빵 리뷰 &amp; 최고의 크보빵 뽑기 + 띠부씰깡까지'),
 ('xSippqKFo5I', '‘Only 타이거즈빵&#39; KIA 크보빵 띠부씰은 몇 개? [실험갸메라]'),
 ('qcgL8Cf8riw', '2025 야구 우승팀, 빵맛으로 정해봤습니다⚾'),
 ('6Eia9PIOx1I', '랜덤깡 민족의 크보빵에서 트윈스 뽑기 [LP]'),
 ('8Yd7WB8GMxg', '3일만에 100만개 팔린 KBO 크보빵'),
 ('9mUWgzwa6Vc', '크보빵 가챠에 도전한 장발장들🦹\u200d♂️🍞'),
 ('nwAhmibEbXM', '크보빵 KBO빵 야구빵 품절대란 9종 간식 먹방 파트2🧁#short'),
 ('e4uWqAiImz4', '역대 최단기록?! 1,000만개가 팔린 크보빵 리뷰 및 국대 띠부씰 뽑기 도전~'),
 ('dddyAu8xUb8', '크보빵으로 불리는 KBO빵 모든 구단 리뷰')]

In [ ]:
# 댓글 추출
video_id = '8Yd7WB8GMxg'

comments = []
request = youtube.commentThreads().list(
    part="snippet",
    videoId=video_id,
    maxResults=10,
    textFormat="plainText"
)
response = request.execute()
for item in response.get("items", []):
    comment = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
    comments.append(comment)

comments

['먹말님 손가락이왔다역시 손가락안나올때부터 보다 손가락나오시는게좋을듯',
 '대놓고 기아만 차별 하자나',
 '저  엘지트윈스 팬이에요',
 '아가리? ㅋㅋ',
 '에휴.. 저거 노동자 갈아서 만든 빵을 왜이리 좋아 하는지\n\n그게 몇년 전부터 툭하면 재발 하는데\n\n그런 기업 제품 왜이리 좋아함?',
 '곰 벌집 터는 맛 ㅋㅋㅋ 오랜만에 왔는데 드립력 여전하시네요',
 '근데 배트빵이 ㄹㅇ 존맛임',
 '빵 퀄리티 보니 개 쓰레기네~',
 '마지막 사잨ㅋㅋ',
 '롯데빵 리뷰 하시져 형님']

In [11]:
# 조회수, 좋아요, 댓글 수 추출
request = youtube.videos().list(
    part="statistics",
    id=video_id
)
response = request.execute()
stats = response["items"][0]["statistics"]

view_count = int(stats.get("viewCount", 0))
like_count = int(stats.get("likeCount", 0))
comment_count = int(stats.get("commentCount", 0))

print(f"조회수: {view_count}")
print(f"좋아요 수: {like_count}")
print(f"댓글 수: {comment_count}")

조회수: 166588
좋아요 수: 2376
댓글 수: 67


## 외부 트렌드 데이터 저장 로직

In [18]:
import os
from dotenv import load_dotenv

# .env 파일의 환경 변수를 로드합니다.
load_dotenv()

# 환경 변수를 변수에 할당합니다.
API_KEY = os.getenv("YOUTUBE_API_KEY")

In [19]:
import json
from googleapiclient.discovery import build

# 1. 설정 (자신의 API 키 입력)
VIDEO_ID = '8Yd7WB8GMxg'
youtube = build("youtube", "v3", developerKey=API_KEY)

def collect_and_save_json(video_id):
    print(f"🚀 영상 ID: {video_id} 데이터 수집 시작...")

    # A. 영상 상세 정보 및 통계 수집
    video_request = youtube.videos().list(
        part="snippet,statistics",
        id=video_id
    )
    video_response = video_request.execute()
    
    if not video_response["items"]:
        print("영상 정보를 찾을 수 없습니다.")
        return

    item = video_response["items"][0]
    snippet = item["snippet"]
    stats = item["statistics"]

    # B. 댓글 수집 (최대 10개)
    comments = []
    try:
        comment_request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=10,
            textFormat="plainText"
        )
        comment_response = comment_request.execute()
        for c_item in comment_response.get("items", []):
            text = c_item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            comments.append(text)
    except Exception as e:
        print(f"댓글 수집 중 오류 발생 (댓글이 비활성화되었을 수 있음): {e}")

    # C. 지식 그래프용 데이터 구조화 (JSON 포맷)
    # nodes_to_extract는 프로젝트의 7개 노드 기준에 맞춰 LLM이 추출할 영역입니다.
    formatted_data = {
        "trend_id": f"YT_{video_id}",
        "keyword": "크보빵",
        "source": "YouTube",
        "video_data": {
            "video_id": video_id,
            "metadata": {
                "title": snippet["title"],
                "channel_title": snippet["channelTitle"],
                "published_at": snippet["publishedAt"]
            },
            "statistics": {
                "view_count": int(stats.get("viewCount", 0)),
                "like_count": int(stats.get("likeCount", 0)),
                "comment_count": int(stats.get("commentCount", 0))
            },
            # 프로젝트의 '7개 노드' 관점에서 태깅 (이후 LLM 처리를 위한 필드)
            "potential_nodes": {
                "IP": ["KBO", "야구"],
                "Product": ["크보빵", "KBO빵"],
                "Category": ["편의점 신상", "간식"],
                "Sentiment_Keywords": ["품절대란", "100만개", "띠부씰"]
            },
            "raw_comments": comments
        }
    }

    # D. JSON 파일로 저장
    file_name = f"youtube_trend_{video_id}.json"
    with open(file_name, 'w', encoding='utf-8') as f:
        json.dump(formatted_data, f, ensure_ascii=False, indent=4)

    print(f"✅ 저장 완료: {file_name}")

In [20]:
if __name__ == "__main__":
    collect_and_save_json(VIDEO_ID)

🚀 영상 ID: 8Yd7WB8GMxg 데이터 수집 시작...
✅ 저장 완료: youtube_trend_8Yd7WB8GMxg.json
